# Tarea ETL — Proceso de Movimientos de inventario (WWImporters)
**Entregable 2 — Implementación del ETL en PySpark**

Este notebook implementa el proceso ETL que puebla en la bodega de datos las dimensiones **Proveedor**, **TipoTransaccion** y **Fecha**, y la **tabla de hechos Movimiento**, a partir de la base transaccional `WWImportersTransactional`.

Las dimensiones **Producto** y **Cliente** NO se procesan aquí: son **dimensiones compartidas (conformadas)** que ya existen en la bodega gracias al proceso de Órdenes; el hecho solo hace *lookup* sobre ellas para traer sus llaves surrogadas.

Cada bloque sigue el flujo **Extraer → Transformar → Cargar** y aplica las reglas de negocio entregadas por WWImporters.

## 0. Configuración y utilidades

In [1]:
# Configuración de conexión
# Use su usuario "Estudiante_i" y la contraseña asignada en el Excel de conexión del curso.
db_user = 'DB_202613_f_cucina'
db_psswd = '202425413'

# Base transaccional (origen) y bodega de datos (destino).
# AJUSTE el nombre de su bodega si es distinto.
BODEGA = 'DB_202613_f_cucina'

source_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/WWImportersTransactional'
dest_db_connection_string   = 'jdbc:mysql://157.253.236.120:8080/' + BODEGA

# Driver de conexión (ajuste la ruta si es necesario)
path_jar_driver = '/workspaces/tareas-modelado-datos-2026-13/drivers/mysql-connector-j.jar'

In [2]:
import os
from pyspark.sql import functions as f, SparkSession, types as t
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql.functions import (col, when, regexp_replace, coalesce, to_timestamp,
                                   to_date, date_format, year, month, dayofmonth, weekofyear)

In [3]:
# Configuración de la sesión Spark
conf = SparkConf().set('spark.driver.extraClassPath', path_jar_driver)
spark_context = SparkContext(conf=conf)
sql_context = SQLContext(spark_context)
spark = sql_context.sparkSession
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
26/06/28 16:49:05 WARN Utils: Your hostname, codespaces-9ec919 resolves to a loopback address: 127.0.0.1; using 10.0.11.58 instead (on interface eth0)
26/06/28 16:49:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 16:49:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
/home/vscode/.local/lib/python3.11/site-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [4]:
# Funciones de conexión: leer de la fuente y guardar en la bodega
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    return spark.read.format('jdbc') \
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()

def guardar_db(db_connection_string, df, tabla, db_user, db_psswd):
    df.select('*').write.format('jdbc') \
        .mode('append') \
        .option('url', db_connection_string) \
        .option('dbtable', tabla) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .save()

## Dimensiones del tutorial

### Dimension Producto

In [8]:
sql_productos = '''(SELECT DISTINCT ID_Producto as ID_Producto_T, ID_Color, NombreProducto, Marca, Necesita_refrigeracion, Dias_tiempo_entrega, Impuesto, PrecioUnitario, PrecioRecomendado FROM WWImportersTransactional.Producto) AS Temp_productos'''
sql_colores = '''(SELECT DISTINCT ID_Color, Color FROM WWImportersTransactional.Colores) AS Temp_colores'''

productos = obtener_dataframe_de_bd(source_db_connection_string, sql_productos, db_user, db_psswd)
colores = obtener_dataframe_de_bd(source_db_connection_string, sql_colores, db_user, db_psswd)

In [9]:
# TRANSFORMACION
productos = productos.join(colores, how = 'left', on = 'ID_Color').fillna({'Color': 'Missing'})
productos = productos.coalesce(1).withColumn('ID_Producto_DWH', f.monotonically_increasing_id() + 1)
productos = productos.select('ID_Producto_DWH','ID_Producto_T','NombreProducto','Marca','Color','Necesita_refrigeracion','Dias_tiempo_entrega','PrecioRecomendado','Impuesto','PrecioUnitario')
productos.show(5)

+---------------+-------------+--------------------+---------+-----+----------------------+-------------------+-----------------+--------+--------------+
|ID_Producto_DWH|ID_Producto_T|      NombreProducto|    Marca|Color|Necesita_refrigeracion|Dias_tiempo_entrega|PrecioRecomendado|Impuesto|PrecioUnitario|
+---------------+-------------+--------------------+---------+-----+----------------------+-------------------+-----------------+--------+--------------+
|              1|           59|RC toy sedan car ...|Northwind|  Red|                     0|                 14|               37|      15|            25|
|              2|           64|RC vintage Americ...|Northwind|  Red|                     0|                 14|               45|      15|            30|
|              3|           68|Ride on toy sedan...|Northwind|  Red|                     0|                 14|              344|      15|           230|
|              4|           73|Ride on vintage A...|Northwind|  Red|        

In [10]:
# CARGUE
guardar_db(dest_db_connection_string, productos,'DB_202613_f_cucina.Producto_t3', db_user, db_psswd)

### Dimension Cliente

In [11]:
sql_categoriasCliente = '''(SELECT DISTINCT ID_Categoria, NombreCategoria FROM WWImportersTransactional.CategoriasCliente) AS Temp_categoriasclientes'''
sql_gruposCompra = '''(SELECT DISTINCT ID_GrupoCompra, NombreGrupoCompra FROM WWImportersTransactional.GruposCompra) AS Temp_gruposcompra'''
sql_clientes = '''(SELECT DISTINCT ID_Cliente as ID_Cliente_T, Nombre, ClienteFactura, ID_Categoria, ID_GrupoCompra, ID_CiudadEntrega, LimiteCredito, FechaAperturaCuenta, DiasPago FROM WWImportersTransactional.Clientes) AS Temp_clientes'''

categoriasCliente = obtener_dataframe_de_bd(source_db_connection_string, sql_categoriasCliente, db_user, db_psswd)
gruposCompra = obtener_dataframe_de_bd(source_db_connection_string, sql_gruposCompra, db_user, db_psswd)
clientes = obtener_dataframe_de_bd(source_db_connection_string, sql_clientes, db_user, db_psswd)

In [12]:
# TRANSFORMACION 
# EL SUPUESTO QUE SE TIENE ES QUE TODOS LOS CLIENTES TIENEN TODOS SUS DATOS DE CATEGORIA Y GRUPO Y NO SE ESTÁN PERDIENDO CLIENTES AL REALIZAR ESTE JOIN
clientes = clientes.join(gruposCompra, how = 'left', on = 'ID_GrupoCompra')
clientes = clientes.alias('cl').join(categoriasCliente.alias('ct'), how = 'left', on = 'ID_Categoria') \
.select([col('cl.ID_Cliente_T'),col('cl.Nombre'),col('ct.NombreCategoria'),col('cl.NombreGrupoCompra') \
        ,col('cl.ClienteFactura'),col('cl.ID_CiudadEntrega'),col('cl.LimiteCredito'),col('cl.FechaAperturaCuenta'),col('cl.DiasPago')])
clientes = clientes.coalesce(1).withColumn('ID_Cliente_DWH', f.monotonically_increasing_id() + 1)
clientes = clientes.select('ID_Cliente_DWH','ID_Cliente_T','Nombre','NombreCategoria','NombreGrupoCompra','ClienteFactura',
                          'ID_CiudadEntrega','LimiteCredito','FechaAperturaCuenta','DiasPago')

clientes = clientes.fillna({'NombreCategoria':'Missing','NombreGrupoCompra':'Missing'})
clientes.show(5)

+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|ID_Cliente_DWH|ID_Cliente_T|              Nombre|NombreCategoria|NombreGrupoCompra|ClienteFactura|ID_CiudadEntrega|LimiteCredito|FechaAperturaCuenta|DiasPago|
+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|             1|         801|         Eric Torres|      Corporate|          Missing|           801|           31321|         3000|2013-01-01 00:00:00|       7|
|             2|         802|        Cosmina Vlad|      Corporate|          Missing|           802|            5192|         2940|2013-01-01 00:00:00|       7|
|             3|         803|          Bala Dixit|   Novelty Shop|          Missing|           803|           33799|         2000|2013-01-01 00:00:00|       7|
|             4|         804|Aleksandrs 

In [13]:
# Crea el registro para el id = 0
clientes_0 = [('0','','Missing','Missing','Missing','0','0','','','')]
columns = ['ID_Cliente_DWH','ID_Cliente_T','Nombre','NombreCategoria','NombreGrupoCompra','ClienteFactura',
            'ID_CiudadEntrega','LimiteCredito','FechaAperturaCuenta','DiasPago']
cliente_0 = spark.createDataFrame(data=clientes_0,schema=columns)
cliente_0.show()

+--------------+------------+-------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|ID_Cliente_DWH|ID_Cliente_T| Nombre|NombreCategoria|NombreGrupoCompra|ClienteFactura|ID_CiudadEntrega|LimiteCredito|FechaAperturaCuenta|DiasPago|
+--------------+------------+-------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|             0|            |Missing|        Missing|          Missing|             0|               0|             |                   |        |
+--------------+------------+-------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+



In [14]:
clientes = clientes.union(cliente_0)
clientes.show(5)

+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|ID_Cliente_DWH|ID_Cliente_T|              Nombre|NombreCategoria|NombreGrupoCompra|ClienteFactura|ID_CiudadEntrega|LimiteCredito|FechaAperturaCuenta|DiasPago|
+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|             1|         801|         Eric Torres|      Corporate|          Missing|           801|           31321|         3000|2013-01-01 00:00:00|       7|
|             2|         802|        Cosmina Vlad|      Corporate|          Missing|           802|            5192|         2940|2013-01-01 00:00:00|       7|
|             3|         803|          Bala Dixit|   Novelty Shop|          Missing|           803|           33799|         2000|2013-01-01 00:00:00|       7|
|             4|         804|Aleksandrs 

In [15]:
# Se ordena por el identificador DWH
clientes = clientes.withColumn('ID_Cliente_DWH',col('ID_Cliente_DWH').cast('int')).orderBy(col('ID_Cliente_DWH'))
clientes.show(5)

+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|ID_Cliente_DWH|ID_Cliente_T|              Nombre|NombreCategoria|NombreGrupoCompra|ClienteFactura|ID_CiudadEntrega|LimiteCredito|FechaAperturaCuenta|DiasPago|
+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|             0|            |             Missing|        Missing|          Missing|             0|               0|             |                   |        |
|             1|         801|         Eric Torres|      Corporate|          Missing|           801|           31321|         3000|2013-01-01 00:00:00|       7|
|             2|         802|        Cosmina Vlad|      Corporate|          Missing|           802|            5192|         2940|2013-01-01 00:00:00|       7|
|             3|         803|          B

In [16]:
# CARGUE
guardar_db(dest_db_connection_string,clientes,'DB_202613_f_cucina.Cliente_t3', db_user, db_psswd)

## BLOQUE 1 — Dimensión Proveedor
Fuente: `proveedores` + `CategoriasProveedores`.

### Extracción

In [5]:
sql_proveedores = """(SELECT DISTINCT ProveedorID, NombreProveedor, CategoriaProveedorID,
       PersonaContactoPrincipalID, DiasPago, CodigoPostal
       FROM WWImportersTransactional.proveedores) AS Temp_proveedores"""

sql_categorias = """(SELECT DISTINCT CategoriaProveedorID, CategoriaProveedor
       FROM WWImportersTransactional.CategoriasProveedores) AS Temp_categorias"""

proveedores = obtener_dataframe_de_bd(source_db_connection_string, sql_proveedores, db_user, db_psswd)
categorias  = obtener_dataframe_de_bd(source_db_connection_string, sql_categorias,  db_user, db_psswd)
proveedores.show(5)

+-----------+--------------------+--------------------+--------------------------+--------+------------+
|ProveedorID|     NombreProveedor|CategoriaProveedorID|PersonaContactoPrincipalID|DiasPago|CodigoPostal|
+-----------+--------------------+--------------------+--------------------------+--------+------------+
|          4|      Fabrikam, Inc.|                   4|                        27|      30|       40351|
|          5|Graphic Design In...|                   2|                        29|      14|       64847|
|          7|       Litware, Inc.|                   5|                        33|      30|       95245|
|          9|      Nod Publishers|                   2|                        37|       7|       27906|
|         10|Northwind Electri...|                   3|                        39|      30|        7860|
+-----------+--------------------+--------------------+--------------------------+--------+------------+
only showing top 5 rows



### Transformación
- **Regla:** unir con `CategoriasProveedores` para traer la categoría.
- **Regla:** los días de pago no pueden ser negativos → multiplicar por -1.
- **Regla:** proveedores con el mismo nombre + sufijo "Inc"/"Ltd" se unifican en uno solo (error de digitación).
- **Regla:** el código postal igual para todos ya fue corregido en la fuente → se toma tal cual.
- Se genera la llave surrogada `ID_Proveedor_DWH`.

In [6]:
# 1. Unir con categorías (left join por CategoriaProveedorID)
proveedores = proveedores.join(categorias, how='left', on='CategoriaProveedorID')

# 2. Corregir días de pago negativos (x -1)
proveedores = proveedores.withColumn(
    'DiasPago', when(col('DiasPago') < 0, col('DiasPago') * -1).otherwise(col('DiasPago')))

# 3. Unificar proveedores con sufijo "Inc"/"Ltd": se limpia el nombre y se eliminan duplicados
proveedores = proveedores.withColumn(
    'NombreProveedor', f.trim(regexp_replace(col('NombreProveedor'), r'\s+(Inc|Ltd)\.?$', '')))
proveedores = proveedores.dropDuplicates(['NombreProveedor'])

# 4. (Código postal ya viene corregido desde la fuente: no requiere transformación)

# 5. Renombrar al esquema de la bodega y generar la llave surrogada
proveedores = proveedores.selectExpr(
    'ProveedorID as ID_Proveedor_T',
    'NombreProveedor as Nombre',
    'CategoriaProveedor as Categoria',
    'PersonaContactoPrincipalID as Contacto_principal',
    'DiasPago as Dias_pago',
    'CodigoPostal as Codigo_postal')
proveedores = proveedores.coalesce(1).withColumn(
    'ID_Proveedor_DWH', (f.monotonically_increasing_id() + 1).cast('int'))
proveedores = proveedores.select('ID_Proveedor_DWH', 'ID_Proveedor_T', 'Nombre',
                                 'Categoria', 'Contacto_principal', 'Dias_pago', 'Codigo_postal')
proveedores.show(5)

+----------------+--------------+--------------------+--------------------+------------------+---------+-------------+
|ID_Proveedor_DWH|ID_Proveedor_T|              Nombre|           Categoria|Contacto_principal|Dias_pago|Codigo_postal|
+----------------+--------------+--------------------+--------------------+------------------+---------+-------------+
|               1|             1| A Datum Corporation| productos novedosos|                21|       14|        46077|
|               2|             3|Consolidated Mess...|servicios de mens...|                25|       30|        94101|
|               3|             2|            Contoso,| productos novedosos|                23|        7|        98253|
|               4|             4|           Fabrikam,|                ropa|                27|       30|        40351|
|               5|             5|Graphic Design In...| productos novedosos|                29|       14|        64847|
+----------------+--------------+---------------

Registro comodín `ID = 0` (representa "sin dato"; lo usa el hecho cuando no hay coincidencia).

In [7]:
prov_0 = spark.createDataFrame([(0, 0, 'Sin dato', 'Sin dato', 0, 0, 0)], schema=proveedores.schema)
proveedores = proveedores.unionByName(prov_0).orderBy(col('ID_Proveedor_DWH'))
proveedores.show(5)

+----------------+--------------+--------------------+--------------------+------------------+---------+-------------+
|ID_Proveedor_DWH|ID_Proveedor_T|              Nombre|           Categoria|Contacto_principal|Dias_pago|Codigo_postal|
+----------------+--------------+--------------------+--------------------+------------------+---------+-------------+
|               0|             0|            Sin dato|            Sin dato|                 0|        0|            0|
|               1|             1| A Datum Corporation| productos novedosos|                21|       14|        46077|
|               2|             3|Consolidated Mess...|servicios de mens...|                25|       30|        94101|
|               3|             2|            Contoso,| productos novedosos|                23|        7|        98253|
|               4|             4|           Fabrikam,|                ropa|                27|       30|        40351|
+----------------+--------------+---------------

### Carga
**OJO:** antes de guardar, asegúrese de que la tabla `Proveedor` esté vacía para no duplicar datos.

In [8]:
guardar_db(dest_db_connection_string, proveedores, BODEGA + '.Proveedor_t3', db_user, db_psswd)

## BLOQUE 2 — Dimensión TipoTransaccion
Fuente: `TiposTransaccion` (ya analizada por el grupo de consultores; se usa tal cual).

### Extracción

In [9]:
sql_tipos = """(SELECT DISTINCT TipoTransaccionID, TipoTransaccionNombre
       FROM WWImportersTransactional.TiposTransaccion) AS Temp_tipos"""
tipos = obtener_dataframe_de_bd(source_db_connection_string, sql_tipos, db_user, db_psswd)
tipos.show()

+-----------------+---------------------+
|TipoTransaccionID|TipoTransaccionNombre|
+-----------------+---------------------+
|                2| Customer Credit Note|
|                3| Customer Payment ...|
|                4|      Customer Refund|
|                5|     Supplier Invoice|
|                6| Supplier Credit Note|
|                7| Supplier Payment ...|
|                8|      Supplier Refund|
|                9|       Stock Transfer|
|               10|          Stock Issue|
|               11|        Stock Receipt|
|               12| Stock Adjustment ...|
|               13|      Customer Contra|
+-----------------+---------------------+



### Transformación
Renombrar al esquema de la bodega y generar la llave surrogada.

In [10]:
tipos = tipos.selectExpr('TipoTransaccionID as ID_Tipo_transaccion_T',
                         'TipoTransaccionNombre as Tipo')
tipos = tipos.coalesce(1).withColumn(
    'ID_Tipo_transaccion_DWH', (f.monotonically_increasing_id() + 1).cast('int'))
tipos = tipos.select('ID_Tipo_transaccion_DWH', 'ID_Tipo_transaccion_T', 'Tipo')

# Registro comodín ID = 0
tipos_0 = spark.createDataFrame([(0, 0, 'Sin dato')], schema=tipos.schema)
tipos = tipos.unionByName(tipos_0).orderBy(col('ID_Tipo_transaccion_DWH'))
tipos.show()

+-----------------------+---------------------+--------------------+
|ID_Tipo_transaccion_DWH|ID_Tipo_transaccion_T|                Tipo|
+-----------------------+---------------------+--------------------+
|                      0|                    0|            Sin dato|
|                      1|                    2|Customer Credit Note|
|                      2|                    3|Customer Payment ...|
|                      3|                    4|     Customer Refund|
|                      4|                    5|    Supplier Invoice|
|                      5|                    6|Supplier Credit Note|
|                      6|                    7|Supplier Payment ...|
|                      7|                    8|     Supplier Refund|
|                      8|                    9|      Stock Transfer|
|                      9|                   10|         Stock Issue|
|                     10|                   11|       Stock Receipt|
|                     11|         

### Carga

In [11]:
guardar_db(dest_db_connection_string, tipos, BODEGA + '.TipoTransaccion_t3', db_user, db_psswd)

## BLOQUE 3 — Dimensión Fecha
Fuente: la columna `FechaTransaccion` de `movimientos` (viene como **texto**).

### Extracción

In [12]:
sql_fechas = """(SELECT DISTINCT FechaTransaccion
       FROM WWImportersTransactional.movimientos) AS Temp_fechas"""
fechas = obtener_dataframe_de_bd(source_db_connection_string, sql_fechas, db_user, db_psswd)

# Revise el formato real del texto para ajustar el parseo si hace falta
fechas.select('FechaTransaccion').show(10, truncate=False)

+----------------+
|FechaTransaccion|
+----------------+
|Jan 20,2014     |
|Jan 28,2014     |
|Feb 01,2014     |
|Mar 25,2014     |
|May 01,2014     |
|May 02,2014     |
|May 10,2014     |
|May 26,2014     |
|Jun 02,2014     |
|Jul 08,2014     |
+----------------+
only showing top 10 rows



### Transformación
- **Regla:** estandarizar el formato (YYYY-MM-DD HH:MM:SS si hay hora; si no, YYYY-MM-DD).
- **Regla:** los años anteriores a 2014 ya están incluidos (no se filtra por año).
- Derivar `Dia`, `Mes`, `Anio`, `Numero_semana_ISO` y generar `ID_Fecha` (entero YYYYMMDD).

In [13]:
# Parseo robusto: intenta varios formatos de texto y se queda con el que funcione
fechas = fechas.withColumn('FechaStd', coalesce(
    to_timestamp(col('FechaTransaccion'), 'yyyy-MM-dd HH:mm:ss'),
    to_timestamp(col('FechaTransaccion'), 'yyyy-MM-dd'),
    to_timestamp(col('FechaTransaccion'), 'MMM dd,yyyy'),
    to_timestamp(col('FechaTransaccion'), 'M/d/yyyy')))

fechas = fechas.withColumn('Fecha', to_date(col('FechaStd')))
fechas = fechas.dropna(subset=['Fecha']).dropDuplicates(['Fecha'])

fechas = fechas \
    .withColumn('ID_Fecha', date_format(col('Fecha'), 'yyyyMMdd').cast('int')) \
    .withColumn('Dia', dayofmonth(col('Fecha'))) \
    .withColumn('Mes', month(col('Fecha'))) \
    .withColumn('Anio', year(col('Fecha'))) \
    .withColumn('Numero_semana_ISO', weekofyear(col('Fecha')))

fechas = fechas.select('ID_Fecha', 'Fecha', 'Dia', 'Mes', 'Anio', 'Numero_semana_ISO')

# Registro comodín ID = 0 (fecha desconocida)
from datetime import date
fecha_0 = spark.createDataFrame([(0, date(1900, 1, 1), 0, 0, 0, 0)], schema=fechas.schema)
fechas = fechas.unionByName(fecha_0).orderBy(col('ID_Fecha'))
fechas.show(5)

+--------+----------+---+---+----+-----------------+
|ID_Fecha|     Fecha|Dia|Mes|Anio|Numero_semana_ISO|
+--------+----------+---+---+----+-----------------+
|       0|1900-01-01|  0|  0|   0|                0|
|20130101|2013-01-01|  1|  1|2013|                1|
|20130102|2013-01-02|  2|  1|2013|                1|
|20130103|2013-01-03|  3|  1|2013|                1|
|20130104|2013-01-04|  4|  1|2013|                1|
+--------+----------+---+---+----+-----------------+
only showing top 5 rows



### Carga

In [14]:
guardar_db(dest_db_connection_string, fechas, BODEGA + '.Fecha_t3', db_user, db_psswd)

## BLOQUE 4 — Tabla de hechos Movimiento
Fuente: `movimientos` + las dimensiones ya cargadas (lookup para traer las llaves surrogadas `_DWH`).

### Extracción

In [5]:
sql_movimientos = """(SELECT DISTINCT ProductoID, ClienteID, ProveedorID, TipoTransaccionID,
       FechaTransaccion, Cantidad
       FROM WWImportersTransactional.movimientos) AS Temp_movimientos"""
mov = obtener_dataframe_de_bd(source_db_connection_string, sql_movimientos, db_user, db_psswd)
print((mov.count(), mov.distinct().count()))
mov.show(5)

(235753, 235753)


+----------+---------+-----------+-----------------+----------------+--------+
|ProductoID|ClienteID|ProveedorID|TipoTransaccionID|FechaTransaccion|Cantidad|
+----------+---------+-----------+-----------------+----------------+--------+
|       108|    185.0|       NULL|               10|     Jan 20,2014|   -10.0|
|       162|      0.0|        4.0|               11|     Jan 28,2014|    10.0|
|       216|    474.0|       NULL|               10|     Jan 28,2014|   -10.0|
|        22|      0.0|        7.0|               11|     Jan 28,2014|    10.0|
|        25|      0.0|        7.0|               11|     Jan 28,2014|    10.0|
+----------+---------+-----------+-----------------+----------------+--------+
only showing top 5 rows



### Transformación
- **Regla:** eliminar las filas totalmente duplicadas (`dropDuplicates`).
- **Regla:** la regla de "máx. 50 millones por transacción" fue eliminada → no se filtra por cantidad.
- **Regla:** las cantidades negativas representan salidas de inventario → se conservan.
- Derivar `ID_Fecha` (YYYYMMDD) a partir de `FechaTransaccion`.
- *Lookup* (left join) a cada dimensión por su llave de negocio (`_T`) para traer las llaves surrogadas (`_DWH`); usar el comodín `0` cuando no haya coincidencia.

In [6]:
# 1. Eliminar duplicados totales
mov = mov.dropDuplicates()

# 2. (No se filtra por cantidad: la regla de 50M fue eliminada)
# 3. (Se conservan las cantidades negativas: son salidas de inventario)

# 4. Derivar ID_Fecha (mismo parseo que en Dim_Fecha)
mov = mov.withColumn('FechaStd', coalesce(
    to_timestamp(col('FechaTransaccion'), 'yyyy-MM-dd HH:mm:ss'),
    to_timestamp(col('FechaTransaccion'), 'yyyy-MM-dd'),
    to_timestamp(col('FechaTransaccion'), 'MMM dd,yyyy'),
    to_timestamp(col('FechaTransaccion'), 'M/d/yyyy')))
mov = mov.withColumn('ID_Fecha', date_format(to_date(col('FechaStd')), 'yyyyMMdd').cast('int'))
mov = mov.fillna({'ID_Fecha': 0})

In [7]:
# 5. Leer las dimensiones desde la bodega para el lookup de llaves surrogadas.
#    Producto y Cliente son dimensiones conformadas que YA existen en la bodega.
dim_producto = obtener_dataframe_de_bd(dest_db_connection_string,
    '(SELECT ID_Producto_DWH, ID_Producto_T FROM ' + BODEGA + '.Producto_t3) AS dp', db_user, db_psswd)
dim_cliente = obtener_dataframe_de_bd(dest_db_connection_string,
    '(SELECT ID_Cliente_DWH, ID_Cliente_T FROM ' + BODEGA + '.Cliente_t3) AS dc', db_user, db_psswd)
dim_proveedor = obtener_dataframe_de_bd(dest_db_connection_string,
    '(SELECT ID_Proveedor_DWH, ID_Proveedor_T FROM ' + BODEGA + '.Proveedor_t3) AS dpr', db_user, db_psswd)
dim_tipo = obtener_dataframe_de_bd(dest_db_connection_string,
    '(SELECT ID_Tipo_transaccion_DWH, ID_Tipo_transaccion_T FROM ' + BODEGA + '.TipoTransaccion_t3) AS dt', db_user, db_psswd)

In [14]:
# Lookups (left join). La llave de negocio del origen se cruza con la _T de cada dimensión.
hecho = mov \
    .join(dim_producto,  mov.ProductoID == dim_producto.ID_Producto_T, 'left') \
    .join(dim_cliente,   mov.ClienteID == dim_cliente.ID_Cliente_T, 'left') \
    .join(dim_proveedor, mov.ProveedorID == dim_proveedor.ID_Proveedor_T, 'left') \
    .join(dim_tipo,      mov.TipoTransaccionID == dim_tipo.ID_Tipo_transaccion_T, 'left')

# Comodín 0 cuando no hubo coincidencia en la dimensión
hecho = hecho.fillna({
    'ID_Producto_DWH': 0, 'ID_Cliente_DWH': 0,
    'ID_Proveedor_DWH': 0, 'ID_Tipo_transaccion_DWH': 0})

# 6. Seleccionar las llaves foráneas + la medida Cantidad
hecho = hecho.select('ID_Fecha', 'ID_Producto_DWH', 'ID_Proveedor_DWH',
                     'ID_Cliente_DWH', 'ID_Tipo_transaccion_DWH', 'Cantidad')
hecho.groupBy("ID_Cliente_DWH").count().orderBy("count", ascending=False).show(5)

+--------------+------+
|ID_Cliente_DWH| count|
+--------------+------+
|             0|117445|
|             4|   254|
|           631|   249|
|            31|   242|
|           293|   240|
+--------------+------+
only showing top 5 rows



In [13]:
m = mov.select(col('ClienteID').cast('int').alias('cid')).distinct()
print("ClienteID distintos en movimientos:", m.count())

for tabla in ['Clientes', 'ClientesProcesados']:
    sql = f"(SELECT DISTINCT ID_Cliente AS cid FROM WWImportersTransactional.{tabla}) AS t"
    d = obtener_dataframe_de_bd(source_db_connection_string, sql, db_user, db_psswd) \
            .select(col('cid').cast('int').alias('cid')).distinct()
    print(tabla, "-> tiene:", d.count(), "| cruzan:", m.join(d,'cid').count())

ClienteID distintos en movimientos: 664


Clientes -> tiene: 663 | cruzan: 663


ClientesProcesados -> tiene: 663 | cruzan: 663


### Carga

In [15]:
guardar_db(dest_db_connection_string, hecho, BODEGA + '.Hecho_Movimiento_t3', db_user, db_psswd)

## Verificación
Verifique en MySQL Workbench que las cuatro tablas (`Proveedor`, `TipoTransaccion`, `Fecha`, `Hecho_Movimiento`) quedaron creadas y pobladas en la bodega `DB_202613_f_cucina`. Recuerde mantener las tablas para que los tutores puedan validarlas.